# MLP vs. KAN: Synthetic Energy-Consumption Prediction

**Google Colab demonstration** — predict building energy consumption from:

- outside temperature \(T\), in °C;
- hour of day \(h\), on a continuous 0–23 scale;
- occupancy \(o\), normalized between 0 and 1.

The notebook compares four models:

1. **MLP 3–4–1** — exactly the architecture in the Beamer slide (21 trainable parameters);
2. **MLP 3–8–1** — a parameter-count-matched reference (41 parameters);
3. **MLP 3–32–1** — a wider one-hidden-layer MLP;
4. **KAN 3–1** — one KAN layer with trainable cubic B-spline functions on the input-to-output edges.

The dataset is synthetic and intentionally uses a sum of smooth, nonlinear one-dimensional effects. This is a good teaching example for the KAN representation, in which each input contributes through a learnable univariate function. **It demonstrates performance on this particular task; it is not proof that KANs outperform MLPs on every dataset or application.**

> Run the cells from top to bottom. Upload `energy_consumption_synthetic.csv` when prompted, or allow the notebook to generate the same dataset automatically.

### Synthetic target function

The target is generated as

\[
E=12+f_T(T)+f_h(h)+f_o(o)+\epsilon,
\qquad \epsilon\sim\mathcal{N}(0,0.25^2),
\]

where

\[
\begin{aligned}
f_T(T)&=0.045(T-20)^2+1.3\sin\!\left(\frac{\pi(T-5)}{30}\right),\\
f_h(h)&=2.4\sin\!\left(\frac{2\pi(h-8)}{24}\right)+0.8\cos\!\left(\frac{4\pi(h-8)}{24}\right),\\
f_o(o)&=5o+2.2o^2.
\end{aligned}
\]

This additive construction is intentional: it provides a controlled example in which learning separate nonlinear functions of the inputs is a natural inductive bias.


## 1. Imports, reproducibility, and execution device

In [ ]:
import os
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)


## 2. Load the synthetic dataset

In [ ]:
CSV_PATH = Path("/content/energy_consumption_synthetic.csv")

# In Colab, upload the CSV supplied with this notebook when it is not already present.
if not CSV_PATH.exists():
    try:
        from google.colab import files
        print("Please select energy_consumption_synthetic.csv.")
        uploaded = files.upload()
        if uploaded:
            CSV_PATH = Path(next(iter(uploaded.keys())))
    except ImportError:
        pass  # Outside Colab: the fallback generator below will be used if needed.

if CSV_PATH.exists():
    data = pd.read_csv(CSV_PATH)
    print(f"Loaded: {CSV_PATH}")
else:
    # Self-contained fallback: reproduces the supplied CSV using the same seed and formula.
    print("CSV not found; generating the reproducible synthetic dataset in memory.")
    rng = np.random.default_rng(SEED)
    n = 6000
    temperature = rng.uniform(5.0, 35.0, n)
    hour = rng.uniform(0.0, 23.0, n)
    occupancy = rng.uniform(0.0, 1.0, n)
    temp_effect = 0.045 * (temperature - 20.0) ** 2 + 1.3 * np.sin(np.pi * (temperature - 5.0) / 30.0)
    hour_effect = 2.4 * np.sin(2.0 * np.pi * (hour - 8.0) / 24.0) + 0.8 * np.cos(4.0 * np.pi * (hour - 8.0) / 24.0)
    occupancy_effect = 5.0 * occupancy + 2.2 * occupancy ** 2
    energy = 12.0 + temp_effect + hour_effect + occupancy_effect + rng.normal(0.0, 0.25, n)
    data = pd.DataFrame({
        "sample_id": np.arange(1, n + 1),
        "temperature_C": temperature,
        "hour_of_day": hour,
        "occupancy": occupancy,
        "energy_kWh": energy,
    })

FEATURES = ["temperature_C", "hour_of_day", "occupancy"]
TARGET = "energy_kWh"
required = FEATURES + [TARGET]
missing = [col for col in required if col not in data.columns]
if missing:
    raise ValueError(f"Missing required dataset columns: {missing}")
if data[required].isna().any().any():
    raise ValueError("Missing values found in features or target. Please clean the CSV first.")

print("Dataset shape:", data.shape)
display(data[required].describe().T)
display(data.head())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(data["temperature_C"], data[TARGET], s=8, alpha=0.35)
axes[0].set_xlabel("Outside temperature (°C)")
axes[0].set_ylabel("Energy consumption (kWh)")
axes[0].set_title("Temperature vs. energy")

axes[1].scatter(data["hour_of_day"], data[TARGET], s=8, alpha=0.35)
axes[1].set_xlabel("Hour of day")
axes[1].set_ylabel("Energy consumption (kWh)")
axes[1].set_title("Hour vs. energy")

axes[2].scatter(data["occupancy"], data[TARGET], s=8, alpha=0.35)
axes[2].set_xlabel("Occupancy (0–1)")
axes[2].set_ylabel("Energy consumption (kWh)")
axes[2].set_title("Occupancy vs. energy")
plt.tight_layout()
plt.show()


## 3. Train / validation / test split and scaling

We use **70% training, 15% validation, and 15% test data**. The scalers are fitted on the training set only. Validation data controls early stopping; the test set is used only for the final reported metrics.

In [ ]:
X = data[FEATURES].to_numpy(dtype=np.float32)
y = data[TARGET].to_numpy(dtype=np.float32)

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=(0.15 / 0.85), random_state=SEED + 1
)

x_scaler = MinMaxScaler(feature_range=(-1, 1), clip=True)
y_scaler = StandardScaler()

X_train_s = x_scaler.fit_transform(X_train).astype(np.float32)
X_val_s = x_scaler.transform(X_val).astype(np.float32)
X_test_s = x_scaler.transform(X_test).astype(np.float32)
y_train_s = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel().astype(np.float32)
y_val_s = y_scaler.transform(y_val.reshape(-1, 1)).ravel().astype(np.float32)

# Convert arrays to tensors once; full-batch training keeps this small demonstration simple.
X_train_t = torch.tensor(X_train_s, dtype=torch.float32, device=DEVICE)
X_val_t = torch.tensor(X_val_s, dtype=torch.float32, device=DEVICE)
y_train_t = torch.tensor(y_train_s, dtype=torch.float32, device=DEVICE)
y_val_t = torch.tensor(y_val_s, dtype=torch.float32, device=DEVICE)

print(f"Train: {len(X_train):,} | Validation: {len(X_val):,} | Test: {len(X_test):,}")


## 4. Define the MLP models

In [ ]:
class MLPRegressor(nn.Module):
    """One-hidden-layer ReLU MLP: input -> hidden -> scalar output."""
    def __init__(self, input_dim=3, hidden_dim=4):
        super().__init__()
        self.hidden = nn.Linear(input_dim, hidden_dim)
        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = F.relu(self.hidden(x))
        return self.output(x).squeeze(-1)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

mlp_341 = MLPRegressor(hidden_dim=4)
print("Slide MLP (3-4-1) trainable parameters:", count_parameters(mlp_341))
assert count_parameters(mlp_341) == 21


## 5. Define a cubic B-spline KAN layer

For this demonstration, the KAN has one output node. Each of the three input-to-output edges learns its own univariate function, represented by cubic B-spline basis functions. The three edge contributions are summed to produce the prediction.

This direct **3→1 KAN** is intentionally aligned with the additive data-generating process. It is a valid, compact KAN architecture and makes the learned edge functions easy to visualize.

In [ ]:
class SplineKANRegressor(nn.Module):
    """One-layer additive KAN with trainable cubic B-spline edge functions.

    Each input feature has a trainable univariate spline to the output node.
    A small SiLU base-function term is included on each edge, as in common KAN
    parameterizations. Inputs are expected to be scaled to approximately [-1, 1].
    """
    def __init__(self, input_dim=3, grid_size=10, degree=3):
        super().__init__()
        self.input_dim = input_dim
        self.grid_size = grid_size
        self.degree = degree
        self.n_basis = grid_size + degree

        # One spline coefficient vector for every input-to-output edge.
        self.spline_weight = nn.Parameter(
            0.03 * torch.randn(1, input_dim, self.n_basis)
        )
        self.base_weight = nn.Parameter(0.05 * torch.ones(1, input_dim))
        self.bias = nn.Parameter(torch.zeros(1))

        step = 2.0 / grid_size
        core = torch.linspace(-1.0, 1.0, grid_size + 1)
        left_extension = core[0] - step * torch.arange(degree, 0, -1)
        right_extension = core[-1] + step * torch.arange(1, degree + 1)
        knots = torch.cat([left_extension, core, right_extension])
        self.register_buffer("knots", knots)

    def bspline_basis(self, x):
        # x: [batch, input_dim] -> basis: [batch, input_dim, n_basis]
        x = x.clamp(-1.0, 1.0).unsqueeze(-1)
        knots = self.knots
        basis = ((x >= knots[:-1]) & (x < knots[1:])).to(x.dtype)

        # Cox–de Boor recursion for B-spline basis functions.
        for p in range(1, self.degree + 1):
            n_basis = knots.numel() - p - 1
            left_den = knots[p:p + n_basis] - knots[:n_basis]
            right_den = knots[p + 1:p + 1 + n_basis] - knots[1:1 + n_basis]
            left_term = (x - knots[:n_basis]) / left_den
            right_term = (knots[p + 1:p + 1 + n_basis] - x) / right_den
            basis = (
                left_term * basis[..., :n_basis]
                + right_term * basis[..., 1:n_basis + 1]
            )
        return basis

    def feature_contributions(self, x):
        """Return each input feature's contribution before adding the bias."""
        basis = self.bspline_basis(x)
        spline_part = (basis * self.spline_weight[0].unsqueeze(0)).sum(dim=-1)
        base_part = F.silu(x) * self.base_weight[0].unsqueeze(0)
        return spline_part + base_part  # [batch, input_dim]

    def forward(self, x):
        contributions = self.feature_contributions(x)
        return contributions.sum(dim=1) + self.bias[0]


kan = SplineKANRegressor(input_dim=3, grid_size=10, degree=3)
print("KAN trainable parameters:", count_parameters(kan))
print("B-spline basis functions per input edge:", kan.n_basis)


## 6. Training function with validation-based early stopping

In [ ]:
def train_model(model, model_name, max_epochs=2000, learning_rate=1e-2, patience=200):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=learning_rate, weight_decay=1e-5
    )
    best_val_loss = float("inf")
    best_state = None
    best_epoch = 0
    epochs_without_improvement = 0
    history = {"train_mse": [], "val_mse": []}

    for epoch in range(1, max_epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        train_pred = model(X_train_t)
        train_loss = F.mse_loss(train_pred, y_train_t)
        train_loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            train_eval_loss = F.mse_loss(model(X_train_t), y_train_t).item()
            val_loss = F.mse_loss(model(X_val_t), y_val_t).item()

        history["train_mse"].append(train_eval_loss)
        history["val_mse"].append(val_loss)

        if val_loss < best_val_loss - 1e-7:
            best_val_loss = val_loss
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            best_epoch = epoch
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    print(
        f"{model_name}: best epoch={best_epoch}, "
        f"validation MSE (standardized target)={best_val_loss:.6f}, "
        f"parameters={count_parameters(model):,}"
    )
    return model, history, best_epoch


## 7. Train all models

In [ ]:
model_specs = [
    ("MLP 3-4-1 (slide)", lambda: MLPRegressor(hidden_dim=4)),
    ("MLP 3-8-1", lambda: MLPRegressor(hidden_dim=8)),
    ("MLP 3-32-1 (wide)", lambda: MLPRegressor(hidden_dim=32)),
    ("KAN 3-1 (cubic splines)", lambda: SplineKANRegressor(input_dim=3, grid_size=10, degree=3)),
]

trained_models = {}
histories = {}
best_epochs = {}

for idx, (name, factory) in enumerate(model_specs):
    # Different, fixed initialization seeds; all models use the same data split and optimizer.
    torch.manual_seed(SEED + idx)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED + idx)
    model, history, best_epoch = train_model(factory(), name)
    trained_models[name] = model
    histories[name] = history
    best_epochs[name] = best_epoch


## 8. Evaluate once on the held-out test set

In [ ]:
X_test_t = torch.tensor(X_test_s, dtype=torch.float32, device=DEVICE)


def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8))) * 100.0
    return {"MSE": mse, "RMSE": rmse, "MAE": mae, "R2": r2, "MAPE (%)": mape}


test_predictions = {}
rows = []
for name, model in trained_models.items():
    model.eval()
    with torch.no_grad():
        pred_scaled = model(X_test_t).detach().cpu().numpy().reshape(-1, 1)
    pred = y_scaler.inverse_transform(pred_scaled).ravel()
    test_predictions[name] = pred
    rows.append({"Model": name, "Parameters": count_parameters(model), **regression_metrics(y_test, pred)})

results = pd.DataFrame(rows).set_index("Model")
results = results.sort_values("RMSE")
print("Final results on the held-out test set (lower error is better; higher R² is better):")
display(results.style.format({
    "MSE": "{:.4f}", "RMSE": "{:.4f}", "MAE": "{:.4f}",
    "R2": "{:.5f}", "MAPE (%)": "{:.3f}"
}))


## 9. Comparison graphs

In [ ]:
# Error metrics: RMSE and MAE
plot_results = results.sort_values("RMSE")
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
plot_results["RMSE"].plot(kind="bar", ax=axes[0], title="Test RMSE (kWh)")
axes[0].set_ylabel("kWh")
axes[0].tick_params(axis="x", rotation=25)
plot_results["MAE"].plot(kind="bar", ax=axes[1], title="Test MAE (kWh)")
axes[1].set_ylabel("kWh")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

# R-squared comparison
fig, ax = plt.subplots(figsize=(9, 4.5))
plot_results["R2"].plot(kind="bar", ax=ax, title="Test R² (higher is better)", ylim=(0, 1.02))
ax.set_ylabel("R²")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()


In [ ]:
# Actual vs. predicted values for every model
fig, axes = plt.subplots(2, 2, figsize=(12, 10), sharex=True, sharey=True)
axes = axes.ravel()
low = min(float(np.min(y_test)), *(float(np.min(p)) for p in test_predictions.values()))
high = max(float(np.max(y_test)), *(float(np.max(p)) for p in test_predictions.values()))
for ax, (name, pred) in zip(axes, test_predictions.items()):
    ax.scatter(y_test, pred, s=12, alpha=0.45)
    ax.plot([low, high], [low, high], linestyle="--", linewidth=1.5)
    ax.set_title(name)
    ax.set_xlabel("Actual energy (kWh)")
    ax.set_ylabel("Predicted energy (kWh)")
plt.tight_layout()
plt.show()


In [ ]:
# Residual distributions on the test set
fig, ax = plt.subplots(figsize=(10, 5))
for name, pred in test_predictions.items():
    residuals = y_test - pred
    ax.hist(residuals, bins=35, alpha=0.35, density=True, label=name)
ax.axvline(0, linestyle="--", linewidth=1.2)
ax.set_xlabel("Residual = actual − predicted (kWh)")
ax.set_ylabel("Density")
ax.set_title("Test residual distributions")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Training and validation curves (MSE on the standardized target)
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, (name, history) in zip(axes.ravel(), histories.items()):
    ax.plot(history["train_mse"], label="Training MSE")
    ax.plot(history["val_mse"], label="Validation MSE")
    ax.axvline(best_epochs[name] - 1, linestyle=":", linewidth=1.2, label="Best epoch")
    ax.set_title(name)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE (standardized target)")
    ax.legend()
plt.tight_layout()
plt.show()


## 10. Visualize the KAN's learned univariate edge functions

Because this KAN is a one-layer additive model, its prediction is the sum of three learned edge functions and a bias. The plots below show each learned contribution in kWh as its corresponding input varies. The other inputs are set to their scaled midpoint for visualization; in this one-layer additive KAN, the plotted edge contribution itself is independent of the other features.

In [ ]:
kan_model = trained_models["KAN 3-1 (cubic splines)"]
scaled_grid = np.linspace(-1.0, 1.0, 250, dtype=np.float32)
feature_names_for_plot = ["Outside temperature (°C)", "Hour of day", "Occupancy (0–1)"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for feature_idx, ax in enumerate(axes):
    grid_matrix = np.zeros((len(scaled_grid), len(FEATURES)), dtype=np.float32)
    grid_matrix[:, feature_idx] = scaled_grid
    grid_tensor = torch.tensor(grid_matrix, dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        contribution_scaled = kan_model.feature_contributions(grid_tensor)[:, feature_idx]
        contribution_scaled = contribution_scaled.detach().cpu().numpy()

    # Undo target standardization for the contribution's vertical scale.
    contribution_kwh = contribution_scaled * float(y_scaler.scale_[0])
    original_x = x_scaler.inverse_transform(grid_matrix)[:, feature_idx]
    ax.plot(original_x, contribution_kwh, linewidth=2)
    ax.set_xlabel(feature_names_for_plot[feature_idx])
    ax.set_ylabel("Learned edge contribution (kWh)")
    ax.set_title(FEATURES[feature_idx])
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 11. Interpretation and teaching points

When presenting the results, focus on these observations:

- The **3–4–1 MLP** has only 21 parameters, exactly as counted in the slide, and may be too restrictive for the three nonlinear effects.
- The **3–8–1 MLP** provides a useful near-parameter-count comparison with the KAN.
- The **wider MLP** tests whether simply increasing the number of ReLU neurons closes the gap.
- The **KAN** learns a separate spline-shaped function for temperature, hour, and occupancy, then sums these contributions. This is particularly suitable for the additive structure used to construct this dataset.
- Compare RMSE and MAE (lower is better), and R² (higher is better). Also inspect actual-vs-predicted plots and residuals rather than relying on one score.

**Important scientific qualification:** these results support the claim that KAN can be more parameter-efficient for this chosen smooth additive problem. They do not establish universal superiority. For a rigorous research comparison, repeat the experiment over several random seeds, noise levels, sample sizes, and datasets with interactions, and report mean ± standard deviation.

## 12. Export test predictions and metrics

In [ ]:
prediction_table = pd.DataFrame({
    "actual_energy_kWh": y_test,
    **{f"pred_{name.replace(' ', '_').replace('-', '')}": pred for name, pred in test_predictions.items()}
})
prediction_file = "mlp_kan_test_predictions.csv"
metrics_file = "mlp_kan_test_metrics.csv"
prediction_table.to_csv(prediction_file, index=False)
results.to_csv(metrics_file)
print("Saved:", prediction_file, "and", metrics_file)

# In Google Colab, uncomment the next two lines to download the result files.
# from google.colab import files
# files.download(metrics_file); files.download(prediction_file)
